In [1]:
# The following code will only execute
# successfully when compression is complete

import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohabenloughmari/miccai-task2-full")

print("Path to dataset files:", path)

Path to dataset files: /root/.cache/kagglehub/datasets/mohabenloughmari/miccai-task2-full/versions/1


In [2]:
!pip install torch==2.5.0 torchvision==0.20.0 torchaudio==2.5.0 --index-url https://download.pytorch.org/whl/cu124

Looking in indexes: https://download.pytorch.org/whl/cu124


In [3]:
!git clone https://github.com/Omid-Nejati/MedViTV2.git

fatal: destination path 'MedViTV2' already exists and is not an empty directory.


In [4]:
!pip install natten==0.17.3+torch250cu124 -f https://shi-labs.com/natten/wheels/ --trusted-host shi-labs.com

Looking in links: https://shi-labs.com/natten/wheels/


In [5]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA version: {torch.version.cuda}")
device=torch.device('cuda')
batch_size = 8


PyTorch version: 2.5.0+cu124
CUDA version: 12.4


In [6]:
import pandas as pd
import torch
from tqdm import tqdm
import torch.nn as nn
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os
import numpy as np
from sklearn.metrics import f1_score, cohen_kappa_score, recall_score, accuracy_score, classification_report
from scipy.stats import spearmanr
from torch.utils.data import WeightedRandomSampler

import warnings
warnings.filterwarnings("ignore")

base_path = path
path = os.path.join(base_path, 'Task_2')

df_train = pd.read_csv(os.path.join(path, "df_task2_train.csv"))
df_val = pd.read_csv(os.path.join(path, "df_task2_val.csv"))
df_test = pd.read_csv(os.path.join(path, "df_task2_test.csv"))

device = torch.device("cuda" if torch.cuda.is_available() else 'cpu')


class CustomImageDataset(Dataset):
    def __init__(self, dataframe, root_dir, transform=None):
        self.dataframe = dataframe
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_name = os.path.join(self.root_dir, self.dataframe.iloc[idx]['image'])
        #localiser_name = os.path.join(self.root_dir, self.dataframe.iloc[idx]['LOCALIZER'])

        image = Image.open(img_name).convert('RGB')
        #localiser = Image.open(localiser_name).convert('RGB')

        if self.transform:
            image = self.transform(image)
            #localiser = self.transform(localiser)

        label = self.dataframe.iloc[idx]['label']
        return image, label


train_transform = transforms.Compose([
    transforms.Resize((336, 336)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((336, 336)),
    transforms.AugMix(alpha= 0.4),
    transforms.RandomHorizontalFlip(p=0.4),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
train_ds = CustomImageDataset(df_train, os.path.join(path, 'train'), transform=train_transform)
val_ds = CustomImageDataset(df_val, os.path.join(path, 'val'), transform=test_transform)
test_ds = CustomImageDataset(df_test, os.path.join(path, 'test'), transform=test_transform)

labels = df_train['label'].values.astype(int)

class_counts = np.bincount(labels)
sample_weights = 1.0 / class_counts[labels]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights))
train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=sampler)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

nb_classes = df_train['label'].nunique()



In [7]:
%cd MedViTV2

/content/MedViTV2


In [8]:
from MedViT import MedViT_tiny, MedViT_small, MedViT_base, MedViT_large
import requests
model_classes = {
    'MedViT_tiny': MedViT_tiny,
    'MedViT_small': MedViT_small,
    'MedViT_base': MedViT_base,
    'MedViT_large': MedViT_large
}

model_urls = {
    "MedViT_tiny": "https://dl.dropbox.com/scl/fi/496jbihqp360jacpji554/MedViT_tiny.pth?rlkey=6hb9froxugvtg8l639jmspxfv&st=p9ef06j8&dl=0",
    "MedViT_small": "https://dl.dropbox.com/scl/fi/6nnec8hxcn5da6vov7h2a/MedViT_small.pth?rlkey=yf5twra1cv6ep2oqr79tbzyg5&st=rwx5hy8z&dl=0",
    "MedViT_base": "https://dl.dropbox.com/scl/fi/q5c0u515dd4oc8j55bhi9/MedViT_base.pth?rlkey=5duw3uomnsyjr80wykvedjhas&st=incconx4&dl=0",
    "MedViT_large": "https://dl.dropbox.com/scl/fi/owujijpsl6vwd481hiydd/MedViT_large.pth?rlkey=cx9lqb4a1288nv4xlmux13zoe&st=kcehwbrb&dl=0"
}

def download_checkpoint(url, path):
    print(f"Downloading checkpoint from {url}...")
    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        with open(path, 'wb') as f:
            for chunk in r.iter_content(chunk_size=8192):
                        f.write(chunk)
    print(f"Checkpoint downloaded and saved to {path}")


In [9]:
# Define the non-MNIST training routine
def specificity_per_class(conf_matrix):
    """Calculates specificity for each class."""
    specificity = []
    for i in range(len(conf_matrix)):
        tn = conf_matrix.sum() - (conf_matrix[i, :].sum() + conf_matrix[:, i].sum() - conf_matrix[i, i])
        fp = conf_matrix[:, i].sum() - conf_matrix[i, i]
        specificity.append(tn / (tn + fp))
    return specificity

def overall_accuracy(conf_matrix):
    """Calculates overall accuracy for multi-class."""
    tp_tn_sum = conf_matrix.trace()  # Sum of all diagonal elements (TP for all classes)
    total_sum = conf_matrix.sum()  # Sum of all elements in the matrix
    return tp_tn_sum / total_sum


In [10]:
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.metrics import f1_score, matthews_corrcoef, confusion_matrix, cohen_kappa_score, classification_report
from sklearn.preprocessing import label_binarize
import sys
from tqdm import tqdm
import torch.optim as optim

state={}
    
def train(epochs, net, train_loader, test_loader, optimizer, scheduler, loss_function, device):
    best_acc = 0.0
    for epoch in range(epochs):
        net.train()
        running_loss = 0.0
        train_bar = tqdm(train_loader, file=sys.stdout) 
        # Training Loop
        for step, datax in enumerate(train_bar):
            images, labels = datax
            images = images.to(device).float()  
            labels = labels.to(device).long() 
            optimizer.zero_grad()
            outputs = net(images.to(device))
            loss = loss_function(outputs, labels.to(device))
            loss.backward()
            optimizer.step()
            scheduler.step()
            running_loss += loss.item() 
            train_bar.desc = f"train epoch[{epoch + 1}/{epochs}] loss:{loss:.3f}"

        # Validation Loop
        net.eval()
        all_preds = []
        all_labels = []
        all_probs = []  # Store raw probabilities/logits for AUC
        acc = 0.0

        with torch.no_grad():
            val_bar = tqdm(test_loader, file=sys.stdout)
            for val_data in val_bar:
                val_images, val_labels = val_data
                outputs = net(val_images.to(device))  # Raw outputs (logits)
                probs = torch.softmax(outputs, dim=1)  # Convert to probabilities

                predict_y = torch.max(probs, dim=1)[1]  # Predicted class   
                # Collect predictions, labels, and probabilities
                all_preds.extend(predict_y.cpu().numpy())
                all_labels.extend(val_labels.cpu().numpy())
                all_probs.extend(probs.cpu().numpy())   
                # Calculate accuracy
                acc += torch.eq(predict_y, val_labels.to(device)).sum().item()

        # Calculate metrics
        val_accurate = acc / len(test_loader.dataset)
        precision = precision_score(all_labels, all_preds, average='weighted')
        recall = recall_score(all_labels, all_preds, average='weighted')  # Sensitivity
        f1 = f1_score(all_labels, all_preds, average='weighted')
        rk_corr = matthews_corrcoef(all_labels, all_preds)
        qwk = cohen_kappa_score(all_labels, all_preds, weights='quadratic')
        # Confusion Matrix for multi-class
        conf_matrix = confusion_matrix(all_labels, all_preds)
        specificity = specificity_per_class(conf_matrix)  # List of specificities per class
        avg_specificity = sum(specificity) / len(specificity)  # Average specificity    
        # Overall Accuracy calculation
        overall_acc = overall_accuracy(conf_matrix) 
        # One-hot encode the labels for AUC calculation
        n_classes = len(conf_matrix)
        all_labels_one_hot = label_binarize(all_labels, classes=list(range(n_classes))) 
        try:
            # Compute AUC for multi-class
            auc = roc_auc_score(all_labels_one_hot, all_probs, multi_class='ovr')
        except ValueError:
            auc = float('nan')  # Handle edge case where AUC can't be computed  
        # Print metrics
        print(f'[epoch {epoch + 1}] train_loss: {running_loss / len(train_loader):.3f} '
              f'val_accuracy: {val_accurate:.4f} precision: {precision:.4f} '
              f'recall: {recall:.4f} specificity: {avg_specificity:.4f} '
              f'f1_score: {f1:.4f} auc: {auc:.4f} overall_accuracy: {overall_acc:.4f} '
              f'mcc: {rk_corr:.4f} qwk: {qwk:.4f}')

        print(f'lr: {scheduler.get_last_lr()[-1]:.8f}')
        print('\nClassification Report:')
        print(classification_report(all_labels, all_preds))
        print(f'lr: {scheduler.get_last_lr()[-1]:.8f}')

        # Save best model
        if val_accurate > best_acc:
            print('\nSaving checkpoint...')
            best_acc = val_accurate
            state = {
                'model': net.state_dict(),
                'optimizer': optimizer.state_dict(),
                'lr_scheduler': scheduler.state_dict(),
                'acc': best_acc,
                'epoch': epoch,
            }
            torch.save(state,'state.pt')
    print('Finished Training')  



In [ ]:
model_name='MedViT_small'
pretrained=True
lr = 1e-4


val_num = len(val_ds)
train_num = len(train_ds)
epochs=10
eta = epochs * train_num // batch_size


if model_name in model_classes:
    model_class = model_classes[model_name]
    net = model_class(num_classes=nb_classes).cuda()
    if pretrained:
        checkpoint_url = model_urls.get(model_name)
        if not checkpoint_url:
            raise ValueError(f"Checkpoint URL for model {model_name} not found.")
        download_checkpoint(checkpoint_url, f'./{model_name}.pth')
        checkpoint_path = f'./{model_name}.pth'
        checkpoint = torch.load(checkpoint_path)
        state_dict = net.state_dict()
        for k in ['proj_head.0.weight', 'proj_head.0.bias']:
            if k in checkpoint and checkpoint[k].shape != state_dict[k].shape:
                print(f"Removing key {k} from pretrained checkpoint")
                del checkpoint[k]
        net.load_state_dict(checkpoint, strict=False)

optimizer = optim.AdamW(net.parameters(), lr=lr, betas=[0.9, 0.999], weight_decay=0.05)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=eta, eta_min=5e-6)
class_weights = torch.tensor(1.0 / class_counts, dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)

    

initialize_weights...
Checkpoint downloaded and saved to ./MedViT_small.pth


In [12]:
train(epochs, net, train_loader, val_loader, optimizer, scheduler, loss_function=criterion,device=device)

100%|██████████| 478/478 [02:23<00:00,  3.32it/s]
[epoch 1] train_loss: 1.087 val_accuracy: 0.2836 precision: 0.5810 recall: 0.2836 specificity: 0.6654 f1_score: 0.2992 auc: 0.5029 overall_accuracy: 0.2836 mcc: -0.0031 qwk: -0.0006
lr: 0.00000500

Classification Report:
              precision    recall  f1-score   support

         0.0       0.12      0.11      0.11       351
         1.0       0.73      0.21      0.33      2821
         2.0       0.17      0.69      0.27       650

    accuracy                           0.28      3822
   macro avg       0.34      0.34      0.24      3822
weighted avg       0.58      0.28      0.30      3822

lr: 0.00000500

Saving checkpoint...
100%|██████████| 478/478 [02:19<00:00,  3.42it/s]
[epoch 2] train_loss: 1.077 val_accuracy: 0.3946 precision: 0.5801 recall: 0.3946 specificity: 0.6651 f1_score: 0.4492 auc: 0.5096 overall_accuracy: 0.3946 mcc: -0.0039 qwk: -0.0036
lr: 0.00000500

Classification Report:
              precision    recall  f1-sc

In [13]:
def evaluate(net, data_loader, device):
    net.eval()

    all_preds = []
    all_labels = []
    all_probs = []
    acc = 0.0

    with torch.no_grad():
        eval_bar = tqdm(data_loader, file=sys.stdout)
        for data in eval_bar:
            images, labels = data
            images = images.to(device).float()
            labels = labels.to(device).long()

            outputs = net(images)
            probs = torch.softmax(outputs, dim=1)
            preds = torch.max(probs, dim=1)[1]

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

            acc += torch.eq(preds, labels).sum().item()

    # Accuracy
    accuracy = acc / len(data_loader.dataset)

    # Metrics
    precision = precision_score(all_labels, all_preds, average='weighted')
    recall = recall_score(all_labels, all_preds, average='weighted')
    f1 = f1_score(all_labels, all_preds, average='weighted')
    mcc = matthews_corrcoef(all_labels, all_preds)
    qwk = cohen_kappa_score(all_labels, all_preds, weights='quadratic')

    # Confusion matrix + derived metrics
    conf_matrix = confusion_matrix(all_labels, all_preds)
    specificity = specificity_per_class(conf_matrix)
    avg_specificity = sum(specificity) / len(specificity)
    overall_acc = overall_accuracy(conf_matrix)

    # AUC (multi-class)
    n_classes = len(conf_matrix)
    all_labels_one_hot = label_binarize(all_labels, classes=list(range(n_classes)))

    try:
        auc = roc_auc_score(all_labels_one_hot, all_probs, multi_class='ovr')
    except ValueError:
        auc = float('nan')

    # Print results
    print(f'Accuracy: {accuracy:.4f}')
    print(f'Precision: {precision:.4f}')
    print(f'Recall: {recall:.4f}')
    print(f'Specificity: {avg_specificity:.4f}')
    print(f'F1-score: {f1:.4f}')
    print(f'AUC: {auc:.4f}')
    print(f'Overall Accuracy: {overall_acc:.4f}')
    print(f'MCC: {mcc:.4f}')
    print(f'QWK: {qwk:.4f}')

    print('\nClassification Report:')
    print(classification_report(all_labels, all_preds))

    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'specificity': avg_specificity,
        'f1': f1,
        'auc': auc,
        'overall_accuracy': overall_acc,
        'mcc': mcc,
        'qwk': qwk,
        'confusion_matrix': conf_matrix
    }

In [14]:
checkpoint = torch.load("state.pt", map_location=device)

net.load_state_dict(checkpoint['model'])
net.to(device)
net.eval()
evaluate(net,test_loader,device)


100%|██████████| 714/714 [03:35<00:00,  3.31it/s]
Accuracy: 0.4756
Precision: 0.7156
Recall: 0.4756
Specificity: 0.6567
F1-score: 0.5631
AUC: 0.4913
Overall Accuracy: 0.4756
MCC: -0.0249
QWK: -0.0270

Classification Report:
              precision    recall  f1-score   support

           0       0.09      0.08      0.08       578
           1       0.84      0.54      0.65      4810
           2       0.04      0.28      0.07       321

    accuracy                           0.48      5709
   macro avg       0.32      0.30      0.27      5709
weighted avg       0.72      0.48      0.56      5709



{'accuracy': 0.47556489753021547,
 'precision': 0.715550134435791,
 'recall': 0.47556489753021547,
 'specificity': np.float64(0.6567244078828122),
 'f1': 0.5630781992960586,
 'auc': np.float64(0.49131192828415804),
 'overall_accuracy': np.float64(0.47556489753021547),
 'mcc': np.float64(-0.024906191216745095),
 'qwk': np.float64(-0.027017913816803407),
 'confusion_matrix': array([[  45,  285,  248],
        [ 452, 2580, 1778],
        [   9,  222,   90]])}